# PACE MiniGrid PLR baseline on Kaggle

Select a **GPU**, enable **Internet**, and choose **Save & Run All**. The trainer source is included in this notebook. It creates the training levels and installs the pinned dependencies. You can leave the Kaggle Inputs panel empty.

The notebook runs **PLR**, seed 0, to **30,000 real PPO batches**, without creating checkpoint or model files. It records reward and interactions on every main-loop batch and saves OOD episode results, CV samples, and aggregate metrics. The default evaluation uses the seven verified reference layouts. A paper-layout JSON can be supplied in `OOD_LAYOUTS_JSON` when the additional assets are available.

In [ ]:
# Pinned trainer snapshot used by the remaining cells.
TRAINER_SOURCE_SHA256 = '7bb5075b2ca843e0023a29ac5842c805ea0083895c3cd678c66a880a929429cd'
TRAINER_SOURCE_B64 = (
    '...'
    'AAA8qQAAAAA='
)

In [ ]:
import base64
import hashlib
import io
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path
from time import perf_counter
from textwrap import dedent

source_start = perf_counter()

MODE = 'train'
METHOD = 'plr'  # dr, plr, plr_perp, accel
SEED = 0
OOD_LAYOUTS_JSON = ''  # Optional asset file with the five additional paper grids.
OOD_EVERY = 500
OOD_CURVE_EPISODES = 20
OOD_FINAL_EPISODES = 100
CV_LEVELS = ['16Rooms', 'Labyrinth', 'Labyrinth2', 'Maze', 'Maze2']
CV_LAYOUTS_JSON = ''
FINAL_OOD_REPORTS = []  # Aggregate mode accepts completed PACE and baseline reports.

if MODE not in ('train', 'aggregate'):
    raise ValueError('Set MODE to train or aggregate.')
if METHOD not in ('dr', 'plr', 'plr_perp', 'accel'):
    raise ValueError('Select dr, plr, plr_perp, or accel.')

WORK = Path('/kaggle/working')
runs = WORK / 'pace'
runs.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix=f'{METHOD}-seed{SEED}-{MODE}-', dir=runs))
print(f'Session output: {RUN}', flush=True)
SOURCE = ''  # Leave empty to use the trainer included in this notebook.
REPO = Path(tempfile.mkdtemp(prefix='pace-baselines-source-'))
RUNTIME = Path('/tmp/pace-kaggle')
if SOURCE:
    source = Path(SOURCE)
    if source.is_dir():
        shutil.copytree(source, REPO, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            archive.extractall(REPO)
    print(f'Using explicit trainer source: {source}', flush=True)
else:
    archive_data = base64.b64decode(TRAINER_SOURCE_B64, validate=True)
    if hashlib.sha256(archive_data).hexdigest() != TRAINER_SOURCE_SHA256:
        raise ValueError('The embedded trainer archive failed its checksum. Re-import the notebook.')
    with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
        archive.extractall(REPO)
    print(f'Using embedded trainer source: {TRAINER_SOURCE_SHA256}', flush=True)

trainer = REPO / 'train/pace_minigrid.py'
if not (REPO / 'scripts/setup_kaggle.sh').is_file() or not trainer.is_file():
    raise FileNotFoundError('The source must contain scripts/setup_kaggle.sh and train/pace_minigrid.py.')
if '--final-only' not in trainer.read_text(encoding='utf-8'):
    raise ValueError('The trainer source does not support checkpoint-free training.')

if MODE == 'aggregate':
    env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'JAX_PLATFORMS': 'cpu',
           'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
else:
    # Use Kaggle's unmodified driver environment for this diagnostic.
    smi = shutil.which('nvidia-smi')
    if not smi or subprocess.run([smi, '-L'], check=False).returncode != 0:
        print('nvidia-smi did not list a GPU. Checking the CUDA driver directly.')

    visible_device = os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',', 1)[0]
    env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'CUDA_VISIBLE_DEVICES': visible_device,
           'JAX_PLATFORMS': 'cuda', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
           'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
    # Keep mounted driver libraries while letting the wheels provide the CUDA toolkit.
    paths = ['/usr/local/nvidia/lib64', *os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep)]
    driver_paths = dict.fromkeys(path for path in paths if Path(path).is_absolute()
                                and Path(path).name not in {'compat', 'stubs'}
                                and (Path(path) / 'libcuda.so.1').is_file())
    if driver_paths:
        env['LD_LIBRARY_PATH'] = os.pathsep.join(driver_paths)
    else:
        env.pop('LD_LIBRARY_PATH', None)

    driver_probe = dedent('''
    import ctypes
    cuda = ctypes.CDLL('libcuda.so.1')
    status = cuda.cuInit(0)
    if status != 0:
        raise RuntimeError(f'CUDA driver initialization failed (cuInit={status})')
    count = ctypes.c_int()
    status = cuda.cuDeviceGetCount(ctypes.byref(count))
    if status != 0 or count.value < 1:
        raise RuntimeError(f'No visible CUDA device (status={status}, count={count.value})')
    ''')
    try:
        subprocess.run([sys.executable, '-c', driver_probe], env=env, check=True)
    except subprocess.CalledProcessError:
        raise RuntimeError('Kaggle has no usable CUDA device for this run. Select a GPU accelerator, '
                           'then restart the session or save a new GPU-enabled version.') from None

source_seconds = perf_counter() - source_start

In [ ]:
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log, subprocess.Popen(
        command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace',
    ) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f'{log_path.stem} failed with exit code {code}. See {log_path}')

setup_start = perf_counter()
run_logged(['bash', 'scripts/setup_kaggle.sh'], RUN / 'setup.log')
PACE_PYTHON = str(RUNTIME / 'venv/bin/python')

probe = '''
import importlib.metadata as metadata
import json
import jax
import jax.numpy as jnp
from jaxued.environments import Maze
from train.pace_minigrid import MiniGridConfig
config = MiniGridConfig.baseline('dr')
assert config.method == 'dr' and config.score_type is None
from train.pace_analysis import MiniGridAnalysis
devices = jax.local_devices(backend='gpu')
assert len(devices) == 1, devices
result = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
assert result.device.platform == 'gpu' and float(result[0, 0]) == 16.0
pin = '0f8f1284677375b889e4f13a32c9617cd009f8c4'
origin = json.loads(metadata.distribution('jaxued').read_text('direct_url.json'))
assert origin['vcs_info']['commit_id'] == pin
print({'device': str(devices[0]), 'jax': jax.__version__, 'jaxued_commit': pin, 'environment': Maze.__name__})
'''
if MODE == 'train':
    run_logged([PACE_PYTHON, '-c', probe], RUN / 'gpu-probe.log')
setup_seconds = source_seconds + perf_counter() - setup_start

In [ ]:
import json

if MODE == 'train':
    suite = 'paper' if OOD_LAYOUTS_JSON else 'reference'
    command = [
        PACE_PYTHON, '-m', 'train.pace_minigrid', 'train', '--device', 'gpu',
        '--method', METHOD, '--seed', str(SEED), '--budget', 'ppo', '--final-only',
        '--suite', suite, '--ood-every', str(OOD_EVERY),
        '--curve-episodes', str(OOD_CURVE_EPISODES), '--final-episodes', str(OOD_FINAL_EPISODES),
        '--cv-levels', *CV_LEVELS, '--output-dir', str(RUN),
    ]
    if OOD_LAYOUTS_JSON:
        command += ['--levels-json', OOD_LAYOUTS_JSON]
    if CV_LAYOUTS_JSON:
        command += ['--cv-levels-json', CV_LAYOUTS_JSON]
    run_logged(command, RUN / 'train.log')
    report_path = RUN / 'run_summary.json'
    summary = json.loads(report_path.read_text(encoding='utf-8'))
    if summary['status'] != 'training_complete' or summary['counts']['ppo_batches'] != 30_000:
        raise RuntimeError(f'Run did not reach the expected PPO budget. See {report_path}')
    summary['timing']['setup_seconds'] = round(setup_seconds, 2)
    summary['trainer_source_sha256'] = TRAINER_SOURCE_SHA256 if not SOURCE else None
    report_path.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(summary, indent=2))

In [ ]:
reports = [str(RUN / 'final_ood.json')] if MODE == 'train' else FINAL_OOD_REPORTS
if not reports:
    raise ValueError('Attach completed evaluation outputs and fill FINAL_OOD_REPORTS.')
run_logged([PACE_PYTHON, '-m', 'train.pace_analysis', 'aggregate',
            '--reports', *reports, '--output-dir', str(RUN)], RUN / 'aggregate.log')
print((RUN / 'ood_aggregate.json').read_text(encoding='utf-8'))